# MLOps Assignment 2 — Jeenal's Submission (v2 — W&B defenses)
**Course:** MLOps · M.Tech AI Trimester-2 · IIT Jodhpur  
**Instructor:** Dr. Hardik Jain  

Fine-tunes DistilBERT on Goodreads reviews. Runs on Kaggle T4 GPU. Tracks training with Weights & Biases. Publishes model to Hugging Face Hub.

**This version adds W&B fallbacks:** if normal `wandb.init` fails, the cell automatically retries in anonymous mode (still produces a public, shareable W&B URL). If even that fails, the cell skips W&B and continues with the HF push so you get partial marks.

**Setup:**
- Settings → Accelerator → **GPU T4 x2**
- Settings → Internet → **On**
- Add-ons → Secrets → `WANDB_API_KEY` and `HF_TOKEN`, both **ticked**

In [1]:
import subprocess, shutil, os
for path in ['/root/.netrc', '/root/.config/wandb', '/root/.cache/wandb']:
    if os.path.exists(path):
        if os.path.isfile(path):
            os.remove(path)
        else:
            shutil.rmtree(path, ignore_errors=True)
print('Cleared any cached W&B state.')

# ---------- 1. Load Kaggle Secrets ----------
from kaggle_secrets import UserSecretsClient

user_secrets  = UserSecretsClient()
WANDB_KEY     = user_secrets.get_secret('WANDB_API_KEY')
HF_API_TOKEN  = user_secrets.get_secret('HF_TOKEN')

os.environ['WANDB_API_KEY']   = WANDB_KEY
os.environ['HF_TOKEN']        = HF_API_TOKEN
os.environ['WANDB_SILENT']    = 'false'

# Sanity check the secrets (don't print actual values)
print(f'HF token prefix: {HF_API_TOKEN[:3]} (must be "hf_")')
print(f'WANDB key loaded: {bool(WANDB_KEY)} | length: {len(WANDB_KEY)}')
if not HF_API_TOKEN.startswith('hf_'):
    print('⚠️  HF token does not start with "hf_" — push to Hub will fail.')

# ---------- 2. Imports ----------
import random as rnd
import gzip as gz
import json as js
import requests as req
import torch as th
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    DistilBertTokenizerFast as DBTokenizer,
    DistilBertForSequenceClassification as DBClassifier,
    Trainer as HfTrainer,
    TrainingArguments as HfArgs,
)
import wandb as wb
from huggingface_hub import login as hf_login

dev = 'cuda' if th.cuda.is_available() else 'cpu'
print(f'Device available: {dev}')
if dev == 'cpu':
    print('⚠️  GPU not detected. Enable Kaggle Settings → Accelerator → GPU T4 x2.')

# ---------- 3. Hyperparameters ----------
PRETRAINED   = 'distilbert-base-cased'
SEQ_LEN      = 256
PER_GENRE    = 250                # 200 train + 50 test per genre
LR           = 3e-5
BATCH_TRAIN  = 16
BATCH_EVAL   = 32
EPOCHS       = 3
WARMUP       = 60
WD           = 0.01
SEED         = 2024
rnd.seed(SEED)
th.manual_seed(SEED)

# ---------- 4. Fetch Goodreads reviews ----------
data_urls = {
    'poetry':                 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_poetry.json.gz',
    'children':               'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_children.json.gz',
    'comics_graphic':         'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_comics_graphic.json.gz',
    'fantasy_paranormal':     'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_fantasy_paranormal.json.gz',
    'history_biography':      'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_history_biography.json.gz',
    'mystery_thriller_crime': 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_mystery_thriller_crime.json.gz',
    'romance':                'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_romance.json.gz',
    'young_adult':            'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_young_adult.json.gz',
}

def fetch_genre(url, lines_cap=2500, target_n=PER_GENRE):
    bucket = []
    resp = req.get(url, stream=True)
    with gz.open(resp.raw, 'rt', encoding='utf-8') as fh:
        for idx, ln in enumerate(fh):
            if idx >= lines_cap:
                break
            bucket.append(js.loads(ln)['review_text'])
    return rnd.sample(bucket, min(target_n, len(bucket)))

reviews_by_genre = {}
for g, u in data_urls.items():
    print(f'Fetching {g}...')
    reviews_by_genre[g] = fetch_genre(u)
print(f'All 8 genres fetched, {PER_GENRE} reviews per genre.')

# ---------- 5. Train / test split ----------
SPLIT = int(PER_GENRE * 0.8)
X_tr, y_tr, X_te, y_te = [], [], [], []
for g, revs in reviews_by_genre.items():
    for r in revs[:SPLIT]:
        X_tr.append(r); y_tr.append(g)
    for r in revs[SPLIT:]:
        X_te.append(r); y_te.append(g)
print(f'Train: {len(X_tr)}  |  Test: {len(X_te)}')

# ---------- 6. Genre maps ----------
genre_set    = sorted(set(y_tr))
genre_to_idx = {g: i for i, g in enumerate(genre_set)}
idx_to_genre = {i: g for g, i in genre_to_idx.items()}

# ---------- 7. Tokenization ----------
tok = DBTokenizer.from_pretrained(PRETRAINED)
enc_tr = tok(X_tr, truncation=True, padding=True, max_length=SEQ_LEN)
enc_te = tok(X_te, truncation=True, padding=True, max_length=SEQ_LEN)
y_tr_idx = [genre_to_idx[v] for v in y_tr]
y_te_idx = [genre_to_idx[v] for v in y_te]
print('Tokenization complete.')

# ---------- 8. Torch Dataset wrapper ----------
class ReviewDS(th.utils.data.Dataset):
    def __init__(self, enc, lab):
        self.enc, self.lab = enc, lab
    def __len__(self):
        return len(self.lab)
    def __getitem__(self, i):
        rec = {k: th.tensor(v[i]) for k, v in self.enc.items()}
        rec['labels'] = th.tensor(self.lab[i])
        return rec

ds_tr = ReviewDS(enc_tr, y_tr_idx)
ds_te = ReviewDS(enc_te, y_te_idx)

# ---------- 9. Load pretrained model ----------
clf = DBClassifier.from_pretrained(
    PRETRAINED,
    num_labels=len(idx_to_genre),
    id2label=idx_to_genre,
    label2id=genre_to_idx,
).to(dev)
print(f'Loaded {PRETRAINED} with {len(idx_to_genre)}-way head.')

# ---------- 10. W&B initialization (with anonymous-mode fallback) ----------

def safe_wandb_init():
    """Try normal init; if it fails, retry in anonymous mode."""
    try:
        run = wb.init(
            project='mlops-assignment2-jeenal',
            name='jeenal-distilbert-bookgenres',
            config={
                'pretrained_model':  PRETRAINED,
                'seq_length':        SEQ_LEN,
                'epochs':            EPOCHS,
                'batch_size_train':  BATCH_TRAIN,
                'batch_size_eval':   BATCH_EVAL,
                'learning_rate':     LR,
                'warmup_steps':      WARMUP,
                'weight_decay':      WD,
                'samples_per_genre': PER_GENRE,
                'dataset':           'UCSD Goodreads (8 genres)',
                'platform':          'Kaggle T4',
                'seed':              SEED,
            },
        )
        print(f'✅ W&B initialized (normal mode): {run.url}')
        return run
    except Exception as e:
        print(f'⚠️  Normal W&B init failed: {e}')
        print('   Falling back to anonymous mode (creates a public link anyone can view)...')
        try:
            run = wb.init(
                project='mlops-assignment2-jeenal',
                name='jeenal-distilbert-bookgenres',
                anonymous='allow',
                config={
                    'pretrained_model':  PRETRAINED,
                    'seq_length':        SEQ_LEN,
                    'epochs':            EPOCHS,
                    'batch_size_train':  BATCH_TRAIN,
                    'batch_size_eval':   BATCH_EVAL,
                    'learning_rate':     LR,
                    'warmup_steps':      WARMUP,
                    'weight_decay':      WD,
                    'samples_per_genre': PER_GENRE,
                    'dataset':           'UCSD Goodreads (8 genres)',
                    'platform':          'Kaggle T4',
                    'seed':              SEED,
                },
            )
            print(f'✅ W&B initialized (anonymous mode): {run.url}')
            print('   NOTE: this run is anonymous but the URL is shareable.')
            return run
        except Exception as e2:
            print(f'❌ Both W&B init attempts failed: {e2}')
            print('   Continuing WITHOUT W&B (you will lose ~25 marks).')
            print('   FIX: rotate the WANDB_API_KEY in Kaggle Secrets and re-run.')
            return None

wb_run = safe_wandb_init()
USE_WANDB = wb_run is not None

# ---------- 11. Metric function ----------
def metrics_fn(pred_out):
    y_true = pred_out.label_ids
    y_hat  = pred_out.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(y_true, y_hat),
        'f1':       f1_score(y_true, y_hat, average='weighted'),
    }

# ---------- 12. Trainer config ----------
hf_args = HfArgs(
    output_dir='./model_out',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,
    learning_rate=LR,
    warmup_steps=WARMUP,
    weight_decay=WD,
    logging_dir='./logs',
    logging_steps=25,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    report_to='wandb' if USE_WANDB else 'none',
    run_name='jeenal-distilbert-bookgenres',
    fp16=True,
    seed=SEED,
)

# ---------- 13. Fine-tune ----------
hf_trainer = HfTrainer(
    model=clf,
    args=hf_args,
    train_dataset=ds_tr,
    eval_dataset=ds_te,
    compute_metrics=metrics_fn,
)
print('Starting fine-tuning on T4 GPU (~8-10 min)...')
hf_trainer.train()
print('Fine-tuning complete.')

# ---------- 14. Final evaluation ----------
final_eval = hf_trainer.evaluate()
print(f'Final evaluation: {final_eval}')

if USE_WANDB:
    wb.log({
        'final/loss':     final_eval['eval_loss'],
        'final/accuracy': final_eval['eval_accuracy'],
        'final/f1':       final_eval['eval_f1'],
    })

# ---------- 15. Classification report → JSON → W&B Artifact ----------
pred_obj  = hf_trainer.predict(ds_te)
y_hat_idx = pred_obj.predictions.argmax(-1)
y_real    = [rec['labels'].item() for rec in ds_te]

cls_rep = classification_report(
    y_real, y_hat_idx,
    target_names=[idx_to_genre[i] for i in sorted(idx_to_genre)],
    output_dict=True,
)

with open('eval_report.json', 'w') as fh:
    js.dump(cls_rep, fh, indent=2)
print('Eval report saved to eval_report.json')

if USE_WANDB:
    art = wb.Artifact('eval-report', type='evaluation')
    art.add_file('eval_report.json')
    wb.log_artifact(art)
    print('Eval report logged as W&B Artifact.')

# ---------- 16. Publish to Hugging Face Hub ----------
hf_login(token=HF_API_TOKEN)

HF_USER = 'jeenal1796'
HF_PATH = f'{HF_USER}/distilbert-bookgenre-classifier'

clf.push_to_hub(HF_PATH)
tok.push_to_hub(HF_PATH)

hf_link = f'https://huggingface.co/{HF_PATH}'
print(f'Model published: {hf_link}')

# ---------- 17. Cross-link HF model in W&B ----------
if USE_WANDB:
    wb.run.summary['huggingface_model'] = hf_link
    wb.run.summary['final_accuracy']    = final_eval['eval_accuracy']
    wb.run.summary['final_f1']          = final_eval['eval_f1']
    wb.run.summary['final_loss']        = final_eval['eval_loss']

# ---------- 18. Close W&B run ----------
if USE_WANDB:
    wb.finish()

print('\n' + '*' * 60)
print('JEENAL — PIPELINE FINISHED')
print('*' * 60)
if USE_WANDB:
    print(f'W&B run page:  {wb_run.url}')
else:
    print('W&B run page:  (W&B was skipped — see fix instructions above)')
print(f'HF model page: {hf_link}')
print(f'Accuracy:      {final_eval["eval_accuracy"]:.4f}')
print(f'F1 (weighted): {final_eval["eval_f1"]:.4f}')
print(f'Eval loss:     {final_eval["eval_loss"]:.4f}')
print('*' * 60)


Cleared any cached W&B state.
HF token prefix: hf_ (must be "hf_")
WANDB key loaded: True | length: 86
Device available: cuda
Fetching poetry...
Fetching children...
Fetching comics_graphic...
Fetching fantasy_paranormal...
Fetching history_biography...
Fetching mystery_thriller_crime...
Fetching romance...
Fetching young_adult...
All 8 genres fetched, 250 reviews per genre.
Train: 1600  |  Test: 400


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenization complete.


...

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Model published: https://huggingface.co/jeenal1796/distilbert-bookgenre-classifier


eval/accuracy,▁▇██
eval/f1,▁▇██
eval/loss,█▂▁▁
eval/runtime,▁▅█▇
eval/samples_per_second,█▄▁▂
eval/steps_per_second,█▄▁▂
final/accuracy,▁
final/f1,▁
final/loss,▁
test/accuracy,▁
+10,...



************************************************************
JEENAL — PIPELINE FINISHED
************************************************************
W&B run page:  https://wandb.ai/jeenalchaudhary1796-education/mlops-assignment2-jeenal/runs/15x8dwb6
HF model page: https://huggingface.co/jeenal1796/distilbert-bookgenre-classifier
Accuracy:      0.5450
F1 (weighted): 0.5431
Eval loss:     2.7250
************************************************************
